# Generative AI Project

### AI-Powered Order Analytics Chatbot using Retrieval-Augmented Generation (RAG)

Leveraging Sentence Transformers, FAISS Indexing, and Gemini API for Intelligent Query-Based Insights from Order Data

In [2]:
path = '/content/order.pdf'

In [3]:
# libraries to install and to put the names in required .txt file
!pip install tabula-py  sentence_transformers faiss-cpu google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.6 MB/s eta 0:00:00


In [4]:
# import packages
import tabula
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from google.colab import userdata

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [5]:
# Load Gemini key from secrets
Gemini_API_Key = userdata.get('Gemini_API')
if Gemini_API_Key is None:
  raise ValueError("No Gemini API Key found in Colab Secrets")

genai.configure(api_key=Gemini_API_Key)

In [6]:
# extracting the tables from pdf
tables = tabula.read_pdf(path, pages='all')
df = pd.concat(tables, ignore_index=True)
df

,order_id,order_status,customer,order_date,order_quantity,sales
0,3,Order\rFinished,Muhammed Mac\rIntyre,13/10/2010,6,523080
1,293,Order\rFinished,Barry French,1/10/2012,49,20246040
2,483,Order\rFinished,Clay Rozendal,10/7/2011,30,9931519
3,515,Order\rFinished,Carlos Soltero,28/8/2010,19,788540
4,613,Order\rFinished,Carl Jackson,17/6/2011,12,187080
5,643,Order\rFinished,Monica Federle,24/3/2011,21,5563640
6,678,Order\rReturned,Dorothy Badders,26/2/2010,44,456820
7,807,Order\rFinished,Neola Schneider,23/11/2010,45,393700
8,868,Order\rFinished,Carlos Daly,8/6/2012,32,1433680
9,933,Order\rFinished,Claudia Miner,4/8/2012,15,161220


This step converts unstructured PDF data into structured tabular format. It enables further analysis and processing by transforming raw order records into a machine-readable dataset.    
The DataFrame represents the complete dataset of customer orders. It serves as the base dataset for generating insights and building the chatbot knowledge base.

In [7]:
print(type(df))

<class 'pandas.core.frame.DataFrame'>


In [8]:
meta_data = df.describe(include = 'all').fillna("")
meta_data

,order_id,order_status,customer,order_date,order_quantity,sales
count,20.0,20,20,20,20.0,20.0
unique,,2,16,19,,
top,,Order\rFinished,Carlos Soltero,4/8/2012,,
freq,,19,3,2,,
mean,1003.65,,,,27.3,4032125.35
std,514.490272,,,,13.010522,6859259.145341
min,3.0,,,,6.0,118060.0
25%,635.5,,,,15.75,342045.0
50%,964.0,,,,26.5,764750.0
75%,1443.75,,,,35.75,4114145.0


This provides an overview of the dataset, including distribution of categorical and numerical features. It helps understand:

- Data completeness
- Variability
- Possible anomalies

In [9]:
# converting dataframe rows to text
documents =[]
for _, row in df.iterrows():
  doc = (
      f"Order ID: {row['order_id']}has status{row['order_status']}\n"
      f"Customer is {row['customer']}. "
      f"Order Date is {row['order_date']}. "
      f"Quantity Ordered is {row['order_quantity']}. "
      f"Total Sales amount is {row['sales']}. "
  )
  documents.append(doc)

print(documents)

['Order ID: 3has statusOrder\rFinished\nCustomer is Muhammed Mac\rIntyre. Order Date is 13/10/2010. Quantity Ordered is 6. Total Sales amount is 523080. ', 'Order ID: 293has statusOrder\rFinished\nCustomer is Barry French. Order Date is 1/10/2012. Quantity Ordered is 49. Total Sales amount is 20246040. ', 'Order ID: 483has statusOrder\rFinished\nCustomer is Clay Rozendal. Order Date is 10/7/2011. Quantity Ordered is 30. Total Sales amount is 9931519. ', 'Order ID: 515has statusOrder\rFinished\nCustomer is Carlos Soltero. Order Date is 28/8/2010. Quantity Ordered is 19. Total Sales amount is 788540. ', 'Order ID: 613has statusOrder\rFinished\nCustomer is Carl Jackson. Order Date is 17/6/2011. Quantity Ordered is 12. Total Sales amount is 187080. ', 'Order ID: 643has statusOrder\rFinished\nCustomer is Monica Federle. Order Date is 24/3/2011. Quantity Ordered is 21. Total Sales amount is 5563640. ', 'Order ID: 678has statusOrder\rReturned\nCustomer is Dorothy Badders. Order Date is 26/2/2

This step transforms structured data into textual format, making it suitable for embedding models. It is crucial because language models understand text better than raw tables.

In [10]:
documents

['Order ID: 3has statusOrder\rFinished\nCustomer is Muhammed Mac\rIntyre. Order Date is 13/10/2010. Quantity Ordered is 6. Total Sales amount is 523080. ',
 'Order ID: 293has statusOrder\rFinished\nCustomer is Barry French. Order Date is 1/10/2012. Quantity Ordered is 49. Total Sales amount is 20246040. ',
 'Order ID: 483has statusOrder\rFinished\nCustomer is Clay Rozendal. Order Date is 10/7/2011. Quantity Ordered is 30. Total Sales amount is 9931519. ',
 'Order ID: 515has statusOrder\rFinished\nCustomer is Carlos Soltero. Order Date is 28/8/2010. Quantity Ordered is 19. Total Sales amount is 788540. ',
 'Order ID: 613has statusOrder\rFinished\nCustomer is Carl Jackson. Order Date is 17/6/2011. Quantity Ordered is 12. Total Sales amount is 187080. ',
 'Order ID: 643has statusOrder\rFinished\nCustomer is Monica Federle. Order Date is 24/3/2011. Quantity Ordered is 21. Total Sales amount is 5563640. ',
 'Order ID: 678has statusOrder\rReturned\nCustomer is Dorothy Badders. Order Date is 

In [11]:
# create embeddings
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, convert_to_numpy=True, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
print(embeddings)

[[-0.0409783   0.00019491  0.07787332 ... -0.07143003 -0.06718098
   0.04090094]
 [-0.03726728 -0.00235723  0.00902483 ... -0.04407313 -0.08012943
   0.00781109]
 [-0.06014673  0.00468356  0.02231907 ... -0.02480754 -0.09258532
  -0.01090562]
 ...
 [-0.00056562  0.01395442  0.08631381 ... -0.03144763 -0.05303966
  -0.01897491]
 [-0.08797383  0.02470109  0.05236213 ... -0.03897326 -0.03434395
   0.00348115]
 [-0.05844679  0.0024894   0.02263454 ... -0.01897111 -0.11053603
   0.03358241]]


Each order is converted into a high-dimensional vector capturing semantic meaning. This allows similarity-based search instead of keyword matching.

In [13]:
# Store in faiss
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
faiss.write_index(index, 'faiss_index.faiss') # saving index file as well

FAISS enables efficient similarity search across large datasets. It helps quickly find the most relevant orders based on user queries.

In [14]:
# retrieval function
def retrive_context(query, k=3): #Finds top 3 similar records
  query_embedding = embedding_model.encode(query).reshape(1, -1) # Reshape for FAISS
  distances, indices = index.search(query_embedding, k)
  return "\n".join([documents[i] for i in indices[0]])

This is the core of RAG. Instead of sending all data to the model, only the most relevant context is retrieved, improving:

- Accuracy
- Speed
- Relevance of responses

In [15]:
# gemini model configuration
# maximum output tokens;
# temp() -> helps model to create response as creative or deterministic, [0,1], the higher the temp--the higher the creativity

generation_config = {
    "temperature":0.4,
    "max_output_tokens": 512
}
print(generation_config)

{'temperature': 0.4, 'max_output_tokens': 512}


In [16]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

In [17]:
gemini_model = genai.GenerativeModel(model_name='models/gemini-2.5-flash', generation_config=generation_config)


Gemini is used as the language model to generate human-like responses. It uses retrieved context + user query to produce meaningful answers.

In [18]:
# conversational bot
chat_history = []
def chat_with_bot(user_input):
    global chat_history

    context = retrive_context(user_input)
    prompt = f"""
    You are an helpful and expert data analyst conversational bot assistant. Please refer to the context below and answer the user's question.
    Context:
    {context}
    User's Question:
    {user_input}

    Rules:
    - Be conversational
    - Answer only using the context
    - If you don't know the answer, say you don't have enough information
    """
    response = gemini_model.generate_content(prompt)
    answer = response.text
    chat_history.append({"user": user_input, "bot":answer})
    return answer

Chatbot is doing:
- Takes user input
- Retrieves relevant context
- Sends prompt to Gemini
- Returns response

Implications:    
This integrates retrieval + generation to create an intelligent conversational system capable of answering questions about order data.

In [19]:
# Checking the working
print("Order Analytics Chat Bot Ready !")
print("Type 'exit' to stop\n")

while True:
  user_input = input("User: ")
  if user_input.lower() in[ 'exit', 'quit', 'bye']:
      print("Good Bye !")
      break
  response = chat_with_bot(user_input)
  print(f"Bot: {response}")
  print("-"*60)

Order Analytics Chat Bot Ready !
Type 'exit' to stop

User: What's the order for Barry?
Bot: Hello there!

I can help you with that.

Barry French's order has an ID of **293**. It was placed on **1/10/2012**, and the quantity ordered was **49**, totaling **20246040** in sales. The order status is **OrderFinished**.
------------------------------------------------------------
User: Did anyone order on 1/11/2012?
Bot: Based on the information provided, I don't see any orders placed on 1/11/2012.
------------------------------------------------------------
User: Is there a customer named Annie?
Bot: Yes, based on the information provided, there is a customer named Annie. Her full name is Annie Cyprus.
------------------------------------------------------------
User: Exit
Good Bye !


This is the final application interface where users can:

- Ask questions (e.g., "Which orders are delayed?")
- Get AI-generated responses based on actual data

#### **`This project demonstrates how structured order data can be transformed into an intelligent conversational system using embeddings, semantic search, and generative AI.`**